In [1]:
import gromacs

In [2]:
gromacs.release()

In [16]:
import MDAnalysis as mda
from MDAnalysis.analysis.distances import self_distance_array
import numpy as np
from imdclient.IMD import IMDReader
import socket 
import time
import sys

/home/jpr122/anaconda3/envs/gsoc/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
gromacs.mdrun_imd(s='seg.tpr', o='seg.trr', c='seg.gro', e='seg.edr',
                  cpo='seg.cpt', g='seg.log', nt=1, imdwait=True, imdport=0)


      :-) GROMACS - gmx mdrun, 2026.0-dev-20250328-3d0839e981-unknown (-:

Executable:   /home/jpr122/Installed_programs/gromacs/build/bin/gmx
Data prefix:  /home/jpr122/Installed_programs/gromacs (source tree)
Working dir:  /home/jpr122/projects/gsoc_westpa_streaming
Command line:
  gmx_imd mdrun -s seg.tpr -o seg.trr -c seg.gro -e seg.edr -cpo seg.cpt -g seg.log -nt 1 -imdwait -imdport 0


Back Off! I just backed up seg.log to ./#seg.log.1#
Reading file seg.tpr, VERSION 2026.0-dev-20250328-3d0839e981-unknown (single precision)
Can not increase nstlist because verlet-buffer-tolerance is not set or used
Using 1 MPI thread
Using 1 OpenMP thread 


NOTE: Thread affinity was not set.

IMD: Enabled. This simulation will accept incoming IMD connections.

Back Off! I just backed up imdforces.xvg to ./#imdforces.xvg.1#
IMD: Pausing simulation while no IMD connection present (-imdwait).
IMD: Setting port for connection requests to 0.
IMD: Setting up incoming socket.
IMD: Listening for IMD conn

KeyboardInterrupt: 



Received the INT signal, stopping within 100 steps




NOTE: 2 % of the run time was spent in domain decomposition,
      24 % of the run time was spent in pair search,
      you might want to increase nstlist (this has no effect on accuracy)

               Core t (s)   Wall t (s)        (%)
       Time:        1.883        1.883      100.0
                 (ns/day)    (hour/ns)
Performance:        1.010       23.774

GROMACS reminds you: "Ease Myself Into the Body Bag" (P.J. Harvey)



In [14]:
import subprocess
import re

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis.distances import self_distance_array
import numpy as np
from imdclient.IMD import IMDReader
import socket 
import time

proc =subprocess.Popen(["gmx_imd", "mdrun", "-s", "seg.tpr", 
                        "-o", "seg.trr", "-c", "seg.gro", "-e", "seg.edr", 
                        "-cpo", "seg.cpt", "-g", "seg.log", 
                        "-nt", "1", "-imdwait", "-imdport", "0"]
                        ,stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
assigned_port = None

for line in proc.stdout:
    print(line, end="") 
    m = re.search(r'IMD connection on port (\d+)', line)
    if m:
        assigned_port = int(m.group(1))
        break
print(f"Assigned IMD port: {assigned_port}")

port=assigned_port
port_open = False
timeout=0.2
host="localhost"
sock = socket.socket(socket.AF_INET,socket.SOCK_STREAM)
while not port_open:
#     sock.settimeout(timeout)
    try:
        sock.connect((host, port))
    except ConnectionRefusedError:
        time.sleep(timeout)
    else:
        print(f"Port {port} on {host} is now open!")
        port_open = True

# sock.close()

u = mda.Universe('bstate.gro', f"imd://localhost:{port}")
ag = u.select_atoms('not water')

dist = []
for ts in u.trajectory:
    dist.append(self_distance_array(ag)[0])
# print(np.array(dist).T)
np.savetxt("dist.dat", np.array(dist))

      :-) GROMACS - gmx mdrun, 2026.0-dev-20250328-3d0839e981-unknown (-:

Executable:   /home/jpr122/Installed_programs/gromacs/build/bin/gmx
Data prefix:  /home/jpr122/Installed_programs/gromacs (source tree)
Working dir:  /home/jpr122/projects/gsoc_westpa_streaming
Command line:
  gmx_imd mdrun -s seg.tpr -o seg.trr -c seg.gro -e seg.edr -cpo seg.cpt -g seg.log -nt 1 -imdwait -imdport 0


Back Off! I just backed up seg.log to ./#seg.log.1#
Reading file seg.tpr, VERSION 2026.0-dev-20250328-3d0839e981-unknown (single precision)
Changing nstlist from 10 to 50, rlist from 0.9 to 0.985

Using 1 MPI thread
Using 1 OpenMP thread 


NOTE: Thread affinity was not set.

IMD: Enabled. This simulation will accept incoming IMD connections.

Back Off! I just backed up imdforces.xvg to ./#imdforces.xvg.1#
IMD: Pausing simulation while no IMD connection present (-imdwait).
IMD: Setting port for connection requests to 0.
IMD: Setting up incoming socket.
IMD: Listening for IMD connection on port 5001